# Hybrid Dual-Frequency Ticker-Specific HMM

Quy trình kết hợp đánh giá Vĩ mô (Monthly) và Thị trường chung (Daily) để định vị pha biến động, sau đó ép cấu trúc thị trường chung lên từng mã cổ phiếu (Ticker-Specific) và dùng Meta-Classifier dự đoán lợi suất 5 ngày tới.

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.stats import skew, kurtosis, norm
from sklearn.feature_selection import mutual_info_regression
from hmmlearn.hmm import GMMHMM, GaussianHMM
import lightgbm as lgb
import shap
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path('../output/hmm_v3_op1_extended')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục đầu ra được thiết lập tại: {OUTPUT_DIR.resolve()}")

Thư mục đầu ra được thiết lập tại: C:\Users\ADMIN\Desktop\Kaggle\output\hmm_v3_op1_extended


## 1. Tải Dữ Liệu & Tạo Chỉ Báo Thị Trường Đại Diện (VN-Index Proxy)

In [2]:
print("Đang tải dữ liệu hmm_data.csv và m1_vn46.csv...")
df_daily_base = pd.read_csv('../output/hmm_data.csv')
df_daily_base['time'] = pd.to_datetime(df_daily_base['time'])

df_m1 = pd.read_csv('../data/processed/m1_vn46.csv')
df_m1['time'] = pd.to_datetime(df_m1['time']).dt.normalize()

# Tạo Market Proxy từ rổ VN46
market_ret = df_m1.groupby('time')['log_return'].mean().reset_index()
market_ret.columns = ['time', 'vnindex_log_ret']
market_close = df_m1.groupby('time')['close'].mean().reset_index()
market_close.columns = ['time', 'vnindex_close']

df_market = df_daily_base.merge(market_ret, on='time', how='left')
df_market = df_market.merge(market_close, on='time', how='left')
df_market = df_market.dropna(subset=['vnindex_log_ret', 'vnindex_close']).reset_index(drop=True)
df_market['vnindex_vol20'] = df_market['vnindex_log_ret'].rolling(20).std() * np.sqrt(252)

try:
    df_fnb = pd.read_csv('../data/processed/m4_foreign_net_buy_sell.csv')
    df_fnb['time'] = pd.to_datetime(df_fnb['time'])
    df_market = df_market.merge(df_fnb[['time', 'fnb_ratio']], on='time', how='left')
except:
    pass

try:
    df_fx = pd.read_csv('../data/processed/e1_usdvnd.csv')
    df_fx['time'] = pd.to_datetime(df_fx['time'])
    df_market = df_market.merge(df_fx[['time', 'fx_log_ret']], on='time', how='left')
except:
    pass

df_market = df_market.dropna().reset_index(drop=True)
print(f"Kích thước bảng dữ liệu Market gốc: {df_market.shape}")

Đang tải dữ liệu hmm_data.csv và m1_vn46.csv...
Kích thước bảng dữ liệu Market gốc: (2401, 13)


## 2. Bộ Lọc Kiểm Định Kỹ Thuật (Stationarity & Kurtosis)

Mục đích của bước này là đánh giá tính chất toán học của các đặc trưng để chọn lọc đầu vào ổn định cho HMM:
1. **Kiểm định tính dừng (Stationarity):** Sử dụng cả ADF (yêu cầu bác bỏ giả thuyết không, $p < 0.05$) và KPSS (yêu cầu chấp nhận giả thuyết không, $p \ge 0.05$) để kiểm tra chuỗi dừng thực tế ở dạng $I(0)$. Chuỗi dừng đảm bảo các thuộc tính thống kê không thay đổi theo thời gian, tránh lỗi ước lượng suy biến.
2. **Hệ số nhọn (Excess Kurtosis):** Kiểm tra xem phân phối của biến có đuôi quá dày hay không ($|kurt| < 10$). Phân phối lệch chuẩn quá mức sẽ vi phạm giả định phân phối Gauss ẩn của HMM.

### Bảng giải thích các chỉ số kiểm định:

| Chỉ số (Metric) | Ý nghĩa (Meaning) | Ngưỡng chấp nhận (Threshold) | Tác dụng trong HMM (Purpose in HMM) |
| :--- | :--- | :--- | :--- |
| **ADF (p-value)** | Kiểm định giả thuyết nghiệm đơn vị (Unit Root). Giả thuyết $H_0$: Chuỗi không dừng. | $p < 0.05$ (Bác bỏ $H_0$, chuỗi dừng) | HMM yêu cầu các đặc trưng đầu vào có tính phân phối ổn định qua thời gian để tránh sai số ước lượng. |
| **KPSS (p-value)** | Kiểm định tính dừng xung quanh xu thế. Giả thuyết $H_0$: Chuỗi dừng. | $p \ge 0.05$ (Không bác bỏ $H_0$, chuỗi dừng) | Bổ trợ cho ADF để xác nhận chắc chắn chuỗi dừng (tránh lỗi ngụy tạo từ ADF). |
| **Kurtosis (Hệ số nhọn)** | Đo lường mức độ tập trung của phân phối quanh giá trị trung bình và độ dày của đuôi. | $\vert \text{Kurt} \vert < 10$ | Tránh hiện tượng đuôi quá béo (fat-tails) gây nhiễu cho ước lượng GMM trong HMM. |
| **Skewness (Hệ số lệch)** | Đo lường tính bất đối xứng của phân phối dữ liệu quanh giá trị trung bình. | Ghi nhận mô tả | Đánh giá xu hướng lệch của biến trước khi nạp vào phân phối chuẩn hỗn hợp. |

In [3]:
def check_stationarity(s):
    s = s.dropna()
    if len(s) < 30: return False, np.nan, np.nan, np.nan, np.nan
    p_adf = adfuller(s, autolag='AIC')[1]
    p_kpss = kpss(s, regression='c', nlags='auto')[1]
    kurt = kurtosis(s)
    skw = skew(s)
    is_stat = (p_adf < 0.05) and (p_kpss >= 0.05) and (abs(kurt) < 10)
    return is_stat, p_adf, p_kpss, kurt, skw

daily_pool = [c for c in df_market.columns if c not in ['time', 'vnindex_log_ret', 'vnindex_close', 'vnindex_vol20']]
stat_results = []
for c in daily_pool:
    is_stat, p_a, p_k, kurt, skw = check_stationarity(df_market[c])
    stat_results.append({'feature': c, 'is_stationary': is_stat, 'p_adf': p_a, 'p_kpss': p_k, 'kurtosis': kurt, 'skewness': skw})
stat_df = pd.DataFrame(stat_results)
display(stat_df)

selected_raw_features = stat_df[stat_df['is_stationary']]['feature'].tolist()
if not selected_raw_features:
    selected_raw_features = daily_pool # Fallback

,feature,is_stationary,p_adf,p_kpss,kurtosis,skewness
0,amihud_diff_normalized,False,7.838500e-21,0.100000,13.484781,-1.175859
1,ret_disp,False,8.902371e-04,0.010000,-0.314023,0.355491
2,rolling_vol_5,True,7.749627e-06,0.062578,6.923895,2.216051
3,volume_ratio,True,1.728272e-21,0.100000,0.479008,0.678426
4,credit_growth_mom,True,1.421514e-15,0.100000,-0.284779,0.298152
5,cpi_mom,True,1.364388e-10,0.100000,4.221535,0.908839
6,pmi_vn,False,8.687660e-02,0.100000,-0.926826,-0.058253
7,fnb_ratio,True,1.225328e-06,0.100000,-0.078439,0.100474
8,fx_log_ret,False,6.613365e-16,0.100000,10.205862,0.110684


## 3. Thiết Lập Tập Dữ Liệu Train/OOS & Hàm Hỗ Trợ HMM sử dụng NQT + Rank

In [4]:
HMM_TRAIN_END = pd.Timestamp('2019-12-31')

def make_Z(df, features, window=252):
    fd = df[['time'] + features].dropna().reset_index(drop=True)
    nqt_df = pd.DataFrame(index=fd.index)
    for col in features:
        rolling_rank = fd[col].rolling(window=window, min_periods=1).rank()
        rolling_count = fd[col].rolling(window=window, min_periods=1).count()
        pct = (rolling_rank - 0.5) / rolling_count
        nqt_values = norm.ppf(pct)
        nqt_df[col] = np.clip(nqt_values, -3.0, 3.0)
    Z_all = nqt_df.values
    train_mask = fd['time'] <= HMM_TRAIN_END
    Z_tr = Z_all[train_mask]
    return fd, Z_tr, Z_all

fd_market, Z_tr_market, Z_all_market = make_Z(df_market, selected_raw_features)
df_market_Z = pd.DataFrame(Z_all_market, columns=[c + '_Z' for c in selected_raw_features])
fd_market = fd_market.merge(df_market[['time', 'vnindex_log_ret', 'vnindex_close', 'vnindex_vol20']], on='time', how='left')
df_market = pd.concat([fd_market, df_market_Z], axis=1)

## 4. Điểm Thông Tin Tương Hỗ (Mutual Information - MI) & Lựa Chọn Đặc Trưng Tham Lam & Kiểm Soát VIF

Điểm MI đo lường mức độ phụ thuộc thông tin (kể cả phi tuyến) giữa đặc trưng đầu vào và trị tuyệt đối lợi suất thị trường `|vnindex_log_ret|` (đại diện cho trạng thái biến động). Điểm số MI cao chỉ ra biến đó có khả năng giải thích cao cho sự thay đổi biến động.

### Bảng giải thích các chỉ số đo lường lượng tin:

| Chỉ số (Metric) | Ý nghĩa (Meaning) | Ngưỡng chấp nhận (Threshold) | Tác dụng trong HMM (Purpose in HMM) |
| :--- | :--- | :--- | :--- |
| **Mutual Information (MI)** | Đo lường lượng thông tin chung thu được về biến mục tiêu thông qua đặc trưng đầu vào (kể cả quan hệ phi tuyến). | Càng cao càng tốt ($\text{MI} > 0.01$ khuyên dùng) | Xác định đặc trưng nào chứa nhiều thông tin giải thích nhất cho trạng thái biến động của thị trường. |

Hàm lọc biến đảm bảo tính đa dạng và hạn chế đa cộng tuyến tuyến tính sử dụng điểm số MI kết hợp hệ số phóng đại phương sai VIF.

### Bảng giải thích chỉ số kiểm soát đa cộng tuyến và đa dạng:

| Chỉ số (Metric) | Ý nghĩa (Meaning) | Ngưỡng chấp nhận (Threshold) | Tác dụng trong HMM (Purpose in HMM) |
| :--- | :--- | :--- | :--- |
| **Variance Inflation Factor (VIF)** | Đo lường mức độ ảnh hưởng của hiện tượng đa cộng tuyến (multicollinearity) giữa một đặc trưng với các đặc trưng khác. | $\text{VIF} < 5.0$ (Tốt); $\text{VIF} \ge 10.0$ (Đa cộng tuyến nghiêm trọng) | Ngăn ngừa ma trận hiệp phương sai của HMM bị suy biến (null eigenvalue) khi huấn luyện mô hình Gauss hỗn hợp. |
| **Block Diversity** | Đảm bảo tính đa dạng thông tin bằng cách lấy ít nhất một biến đại diện từ mỗi khâu (`Market`, `Economy`, `Credit`). | Bắt buộc chọn từ các nhóm khác nhau | Giúp HMM quan sát thị trường từ nhiều khía cạnh bổ trợ thay vì chỉ tập trung vào một nguồn tin. |

In [5]:
# Tạo Y_proxy rule-based cho Market
vol_median = df_market['vnindex_vol20'].median()
def label_proxy(row):
    ret = row['vnindex_log_ret']
    vol = row['vnindex_vol20']
    if ret > 0 and vol < vol_median: return 0 # Bull / Low Vol
    elif ret < 0 and vol > vol_median: return 1 # Bear / High Vol
    else: return 2 # Sideways

df_market['Y_proxy'] = df_market.apply(label_proxy, axis=1)

Z_features = [c + '_Z' for c in selected_raw_features]
X_train = df_market[Z_features].dropna()
y_train = df_market.loc[X_train.index, 'Y_proxy']

# Tính SHAP
clf = lgb.LGBMClassifier(n_estimators=50, random_state=RANDOM_STATE, verbose=-1)
clf.fit(X_train, y_train)
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_train)
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    vals = shap_values.values if hasattr(shap_values, 'values') else np.array(shap_values)
    if len(vals.shape) == 3:
        if vals.shape[1] == len(Z_features):
            mean_shap = np.abs(vals).mean(axis=0).mean(axis=1)
        else:
            mean_shap = np.abs(vals).mean(axis=0).mean(axis=0)
    else:
        mean_shap = np.abs(vals).mean(axis=0)
shap_df = pd.DataFrame({'feature': Z_features, 'shap_importance': mean_shap})

# Tính MI với |vnindex_log_ret|
target_mi = np.abs(df_market.loc[X_train.index, 'vnindex_log_ret'])
mi_scores = mutual_info_regression(X_train, target_mi, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({'feature': Z_features, 'mi_score': mi_scores})

# Gộp điểm & Lọc VIF tham lam
feature_scores = shap_df.merge(mi_df, on='feature')
feature_scores['total_score'] = feature_scores['shap_importance'] * feature_scores['mi_score']
feature_scores = feature_scores.sort_values('total_score', ascending=False)

def filter_vif_greedy(df, features, threshold=5.0):
    selected = []
    for f in features:
        trial = selected + [f]
        X_sub = df[trial].values
        X_sub = np.column_stack([np.ones(len(X_sub)), X_sub])
        vif = variance_inflation_factor(X_sub, len(trial))
        if vif < threshold:
            selected.append(f)
    return selected

final_features = filter_vif_greedy(X_train, feature_scores['feature'].tolist())
print("Top features (SHAP+MI) qua bộ lọc VIF:")
print(final_features)
display(feature_scores.head(10))

Top features (SHAP+MI) qua bộ lọc VIF:
['volume_ratio_Z', 'cpi_mom_Z', 'credit_growth_mom_Z', 'fnb_ratio_Z', 'rolling_vol_5_Z']


,feature,shap_importance,mi_score,total_score
1,volume_ratio_Z,0.141936,0.027493,0.003902
3,cpi_mom_Z,0.281687,0.013354,0.003762
2,credit_growth_mom_Z,0.231244,0.010242,0.002368
4,fnb_ratio_Z,0.091187,0.022305,0.002034
0,rolling_vol_5_Z,0.243549,0.005873,0.001430


## 5. Chạy Grid Search Tìm Cấu HÌnh HMM Tốt Nhất

Tối ưu hóa đồng thời số lượng đặc trưng ($n_{features}$) và số lượng trạng thái ẩn ($K$) cho mô hình ngày, và số trạng thái ẩn vĩ mô tháng. Các cấu hình được đánh giá qua các chỉ số chất lượng để chọn ra mô hình cân bằng nhất.

### Bảng giải thích các chỉ số đánh giá Grid Search HMM:

| Chỉ số (Metric) | Ý nghĩa (Meaning) | Ngưỡng chọn/Ràng buộc (Constraint) | Tác dụng trong tối ưu hóa HMM (Purpose in HMM) |
| :--- | :--- | :--- | :--- |
| **D** | Số lượng chiều (dimensions) của dữ liệu. | Tối ưu theo mô hình | Quyết định xem mỗi bước được quyết định bởi bao nhiêu biến số |
| **ll_in** | Log-Likelihood trong mẫu (In-sample). Đo lường độ khớp của mô hình với tập Train. | Càng cao càng tốt | Đánh giá khả năng giải thích dữ liệu huấn luyện của mô hình. |
| **ll_oos** | Log-Likelihood ngoài mẫu (Out-of-sample). Đo lường độ khớp trên tập kiểm thử chưa thấy ứng với seed tốt nhất. | Càng cao càng tốt | Đánh giá khả năng tổng quát hóa của mô hình tốt nhất chọn được. |
| **avg_ll_oos** | Trung bình Log-Likelihood ngoài mẫu (Average OOS LL) tính trên tất cả các seed hội tụ. | Càng cao càng tốt | Đo lường độ ổn định tổng quát hóa của thuật toán, tránh sự may rủi của một seed khởi tạo duy nhất. |
| **bic** | Chỉ số thông tin Bayesian (Bayesian Information Criterion). Phạt mô hình có quá nhiều tham số. | Càng thấp càng tốt | Lựa chọn số lượng đặc trưng và số trạng thái ẩn tối ưu nhất, ngăn chặn quá khớp (Overfitting). |
| **min_dur** | Thời gian lưu trú tối thiểu ở một trạng thái (Minimum State Duration). | $\ge 3$ phiên giao dịch/tháng | Đảm bảo các trạng thái ẩn có tính bền vững nhất định, tránh việc chuyển đổi trạng thái liên tục gây nhiễu (chattering). |
| **min_share / max_share** | Tỷ lệ số phiên tối thiểu/tối đa thuộc một trạng thái trong tập Train. | $\text{min\_share} \ge 0.05$, $\text{max\_share} \le 0.75$ | Đảm bảo sự phân bổ cân bằng giữa các trạng thái ẩn, tránh trường hợp một trạng thái quá loãng hoặc chiếm hết dữ liệu. |

$$Composite = 0.3 \cdot Rank_{bic} + 0.5 \cdot Rank_{oos} + 0.2 \cdot Rank_{min\_dur}$$

In [12]:
def n_params(K, D, M=2):
    return (K - 1) + K * (K - 1) + K * (M - 1) + K * M * D + K * M * D * (D + 1) // 2

train_mask = df_market['time'] <= HMM_TRAIN_END
Z_data = df_market[final_features].fillna(0).values
Z_train = Z_data[train_mask]
Z_oos = Z_data[~train_mask]

def evaluate_hmm(K, Z_train, Z_oos, seeds=3):
    best_bic, best_ll_oos, best_model = np.inf, -np.inf, None
    best_min_dur, best_min_share, best_max_share = 0, 0, 1
    
    for seed in range(seeds):
        try:
            m = GMMHMM(n_components=K, n_mix=2, covariance_type='full', min_covar=0.01, n_iter=200, random_state=seed*7)
            m.fit(Z_train)
            ll_train = m.score(Z_train)
            
            p = n_params(K, Z_train.shape[1])
            bic = -2 * ll_train + p * np.log(len(Z_train))
            ll_oos = m.score(Z_oos) if len(Z_oos)>0 else np.nan
            
            persist = np.diag(m.transmat_)
            min_dur = float((1.0 / (1.0 - persist + 1e-9)).min())
            
            preds = m.predict(Z_train)
            counts = np.bincount(preds, minlength=K) / len(Z_train)
            min_share = float(counts.min())
            max_share = float(counts.max())
            
            if bic < best_bic and min_dur >= 3.0 and min_share >= 0.05 and max_share <= 0.75:
                best_bic, best_ll_oos, best_model = bic, ll_oos, m
                best_min_dur, best_min_share, best_max_share = min_dur, min_share, max_share
        except:
            continue
    return best_model, best_bic, best_ll_oos, best_min_dur, best_min_share, best_max_share

results, models = [], {}
for K in [2, 3, 4]:
    m, bic, ll_oos, min_dur, min_share, max_share = evaluate_hmm(K, Z_train, Z_oos)
    if m:
        results.append({
            'K': K, 'BIC': bic, 'll_oos': ll_oos, 'min_dur': min_dur,
            'min_share': min_share, 'max_share': max_share
        })
        models[K] = m

res_df = pd.DataFrame(results)
if len(res_df) > 0:
    res_df['Rank_bic'] = res_df['BIC'].rank(ascending=True)
    res_df['Rank_oos'] = res_df['ll_oos'].rank(ascending=False)
    res_df['Rank_min_dur'] = res_df['min_dur'].rank(ascending=False)
    res_df['Composite'] = 0.3 * res_df['Rank_bic'] + 0.5 * res_df['Rank_oos'] + 0.2 * res_df['Rank_min_dur']
    res_df = res_df.sort_values('Composite')
    display(res_df)
    
    K_market =  4 # int(res_df.iloc[0]['K'])
    best_market_hmm = models[K_market]
    print(f"--> Quyết định K tối ưu thị trường: K = {K_market}")
    
    market_probs = best_market_hmm.predict_proba(Z_data)
    for i in range(K_market):
        df_market[f'Market_Prob_{i}'] = market_probs[:, i]
else:
    print("Không tìm thấy cấu hình HMM hội tụ thỏa mãn điều kiện!")

,K,BIC,ll_oos,min_dur,min_share,max_share,Rank_bic,Rank_oos,Rank_min_dur,Composite
0,2,9404.562695,-16832.151175,31.857904,0.403287,0.596713,3.0,1.0,1.0,1.6
1,3,8956.845547,-17169.705931,18.875928,0.217446,0.431100,2.0,2.0,2.0,2.0
2,4,8488.728650,-18698.031729,11.650360,0.146650,0.447535,1.0,3.0,3.0,2.4


--> Quyết định K tối ưu thị trường: K = 4


## 6. Tự Động Ánh Xạ & Gán Nhãn Trạng Thái (K-agnostic Labeling)

Ánh xạ nhãn trạng thái tự động cho cả vĩ mô tháng và thị trường ngày bám sát theo kỳ vọng lợi nhuận kì vọng và biến động tài sản.

### Labling Monthly
| Trạng thái (Status) | Ý nghĩa (Meaning) | Ngưỡng chọn / Ràng buộc (Constraint) |
|---------------------|-------------------|--------------------------------------|
| **Macro_Stagnant** (K=2,3) | Giai đoạn vĩ mô trì trệ, sản xuất suy giảm hoặc tăng trưởng chậm. | `pmi_vn` thấp nhất |
| **Macro_Stable** (K=3) | Giai đoạn vĩ mô ổn định, tăng trưởng sản xuất vừa phải. | `pmi_vn` ở mức trung vị / trung bình |
| **Macro_Expansion** (K=2,3) | Giai đoạn vĩ mô mở rộng, sản xuất tăng trưởng mạnh mẽ. | `pmi_vn` cao nhất |

### Labling Daily
K = 3
| Trạng thái (Status) | Ý nghĩa (Meaning) | Ngưỡng chọn / Ràng buộc (Constraint) |
|---------------------|-------------------|--------------------------------------|
| **Bull** | Thị trường tăng trưởng, xu hướng đi lên và ít biến động (rủi ro thấp). | `ret` không thấp nhất và `vol` thấp nhất trong 2 trạng thái còn lại |
| **Sideways** | Thị trường đi ngang, dao động tích lũy trong biên độ. | Trạng thái còn lại sau khi đã xác định Bull và Bear |
| **Bear** | Thị trường suy thoái, xu hướng giảm mạnh và rủi ro cao. | `ret` thấp nhất (argmin) |

K = 4
| Trạng thái (Status) | Ý nghĩa (Meaning) | Ngưỡng chọn / Ràng buộc (Constraint) |
|---------------------|-------------------|--------------------------------------|
| **Crisis** | Thị trường suy thoái, biến động mạnh, rủi ro cao. | `ret < 0` và `vol >= median` |
| **CalmBull** | Pha tăng trưởng bền vững, ổn định, ít rủi ro. | `ret > 0` và `vol < median` |
| **Euphoria** | Thị trường hưng phấn, tăng mạnh kèm dao động lớn. | `ret > 0` và `vol >= median` |
| **Tranquil** | Thị trường "lặng sóng", đi ngang, ảm đạm. | `ret ≈ 0` và `vol < median` |

In [13]:
df_market_res = df_market[['time']].copy()
df_market_res['market_regime'] = best_market_hmm.predict(Z_data)
stats_market = []
for k in range(K_market):
    mask = df_market_res['market_regime'] == k
    ret_k = df_market.loc[mask, 'vnindex_log_ret'].mean() * 100 if mask.sum() > 0 else 0
    vol_k = df_market.loc[mask, 'vnindex_vol20'].mean() * 100 if mask.sum() > 0 else 0
    stats_market.append({'state': k, 'mean_ret_%': ret_k, 'vol_%': vol_k})
rs_market = pd.DataFrame(stats_market)
display(rs_market)

def auto_label(rs, K):
    ret = rs['mean_ret_%'].values
    vol = rs['vol_%'].values
    if K == 2:
        return {int(np.argmin(ret)): 'Bear', int(np.argmax(ret)): 'Bull'}
    elif K == 3:
        order = np.argsort(ret)
        bear_idx = order[0]
        rem = [order[1], order[2]]
        if vol[rem[0]] < vol[rem[1]]:
            bull_idx, side_idx = rem[0], rem[1]
        else:
            bull_idx, side_idx = rem[1], rem[0]
        return {int(bear_idx): 'Bear', int(side_idx): 'Sideways', int(bull_idx): 'Bull'}
    elif K >= 4:
        from scipy.optimize import linear_sum_assignment
        ret_Z = (ret - np.mean(ret)) / (np.std(ret) + 1e-9)
        vol_Z = (vol - np.mean(vol)) / (np.std(vol) + 1e-9)
        
        scores = np.zeros((K, 4))
        for i in range(K):
            scores[i, 0] = -ret_Z[i] + vol_Z[i] # Crisis
            scores[i, 1] = -ret_Z[i] - vol_Z[i] # Tranquil
            scores[i, 2] =  ret_Z[i] - vol_Z[i] # CalmBull
            scores[i, 3] =  ret_Z[i] + vol_Z[i] # Euphoria
            
        row_ind, col_ind = linear_sum_assignment(-scores)
        label_names = ['Crisis', 'Tranquil', 'CalmBull', 'Euphoria']
        labels = {r: label_names[c] for r, c in zip(row_ind, col_ind)}
        
        unassigned = set(range(K)) - set(row_ind)
        for i, st in enumerate(unassigned):
            labels[st] = f'Daily_Tier{i+5}'
            
        return labels
    return {i: f'State_{i}' for i in range(K)}

STATE_TO_LABEL_MARKET = auto_label(rs_market, K_market)
df_market_res['market_regime_label'] = df_market_res['market_regime'].map(STATE_TO_LABEL_MARKET)
for k in range(K_market):
    df_market_res[f'prob_market_{k}'] = df_market[f'Market_Prob_{k}']

,state,mean_ret_%,vol_%
0,0,-0.000698,18.482888
1,1,-0.121271,18.538166
2,2,0.030077,19.716538
3,3,0.217051,19.479043


## 7. Suy Luận Trạng Thái Ẩn Cho Từng Mã Cổ Phiếu (Ticker-Specific Inference)

In [14]:
def make_Z_ticker(df_source, features, window=252):
    fd = df_source[['time'] + features].dropna().reset_index(drop=True)
    nqt_df = pd.DataFrame(index=fd.index)
    for col in features:
        rolling_rank = fd[col].expanding(min_periods=1).rank()
        rolling_count = fd[col].expanding(min_periods=1).count()
        pct = (rolling_rank - 0.5) / rolling_count
        nqt_values = norm.ppf(pct)
        nqt_df[col] = np.clip(nqt_values, -3.0, 3.0)
    Z_all = nqt_df.values
    return fd, Z_all

tickers = df_m1['ticker'].unique()
print(f'Bắt đầu xử lý suy luận cho {len(tickers)} mã...')

market_cols = [c for c in df_market.columns if 'Z' not in c and c not in ['time', 'Y_proxy']]
global_vars = df_market[['time'] + market_cols].copy()

ticker_dfs = []
for i, ticker in enumerate(tickers):
    df_tick = df_m1[df_m1['ticker'] == ticker].copy().sort_values('time').reset_index(drop=True)
    df_tick['rolling_vol_5'] = df_tick['log_return'].rolling(5).std() * np.sqrt(252)
    df_tick['mom_1M'] = df_tick['close'].pct_change(20)
    df_tick['dist_MA50'] = df_tick['close'] / df_tick['close'].rolling(50).mean() - 1
    
    for col in ['volume_ratio', 'rolling_vol_20d', 'return_5d', 'return_20d']:
        if col not in df_tick.columns:
            if col == 'volume_ratio': df_tick['volume_ratio'] = df_tick['volume'] / df_tick['volume'].rolling(20).mean()
            elif col == 'rolling_vol_20d': df_tick['rolling_vol_20d'] = df_tick['log_return'].rolling(20).std() * np.sqrt(252)
            elif col == 'return_5d': df_tick['return_5d'] = df_tick['close'].pct_change(5)
            elif col == 'return_20d': df_tick['return_20d'] = df_tick['close'].pct_change(20)
    
    ticker_cols = ['time', 'open', 'high', 'low', 'close', 'volume', 'log_return', 
                   'rolling_vol_20d', 'return_5d', 'return_20d', 'volume_ratio', 'rolling_vol_5', 'mom_1M', 'dist_MA50']
    
    cols_to_drop = [c for c in ticker_cols if c != 'time' and c in global_vars.columns]
    global_vars_clean = global_vars.drop(columns=cols_to_drop)
    ticker_aligned = global_vars_clean.merge(df_tick[ticker_cols], on='time', how='inner')
    
    raw_features = [f.replace('_Z', '') for f in final_features]
    tick_features = [f for f in raw_features if f in ticker_aligned.columns]
    
    fd_z_tick, Z_all_tick = make_Z_ticker(ticker_aligned, tick_features, window=252)
    
    # Ép buộc dùng nguyên bản Model của thị trường (Force K_stock = K_market)
    try:
        ticker_daily_states = best_market_hmm.predict(Z_all_tick)
        ticker_daily_probs = best_market_hmm.predict_proba(Z_all_tick)
    except Exception as e:
        print(f"Lỗi khi predict mã {ticker}: {e}")
        ticker_daily_states = np.zeros(len(Z_all_tick), dtype=int)
        ticker_daily_probs = np.zeros((len(Z_all_tick), K_market))
        ticker_daily_probs[:, 0] = 1.0
        
    # Áp dụng trực tiếp bộ nhãn của thị trường để đảm bảo tính đồng nhất
    ticker_daily_labels = pd.Series(ticker_daily_states).map(STATE_TO_LABEL_MARKET).values
    
    df_tick_daily_res = pd.DataFrame({
        'time': fd_z_tick['time'].values,
        'market_regime': ticker_daily_states,
        'market_regime_label': ticker_daily_labels,
    })
    for k in range(K_market):
        df_tick_daily_res[f'prob_market_{k}'] = ticker_daily_probs[:, k]
        
    state_cols = ['market_regime', 'market_regime_label'] + [f'prob_market_{k}' for k in range(K_market)]
    ticker_master = ticker_aligned.merge(df_tick_daily_res[['time'] + state_cols], on='time', how='inner')
    ticker_master['ticker'] = ticker
    
    ticker_dfs.append(ticker_master)

master_ticker = pd.concat(ticker_dfs, ignore_index=True)
master_ticker = master_ticker.dropna(subset=['close']).reset_index(drop=True)
cols_reordered = ['time', 'ticker'] + [col for col in master_ticker.columns if col not in ['time', 'ticker']]
master_ticker = master_ticker[cols_reordered]
print(f'Hoàn thành suy luận. Kích thước master_ticker: {master_ticker.shape}')

Bắt đầu xử lý suy luận cho 46 mã...
Hoàn thành suy luận. Kích thước master_ticker: (110446, 31)


## 8. Lưu Kết Quả Đầu Ra & Chia Tập Dữ Liệu Splits

In [15]:
df_market_res.to_parquet(OUTPUT_DIR / 'daily_market_regimes.parquet', index=False)
master_ticker.to_parquet(OUTPUT_DIR / 'master_drl_ready_ticker.parquet', index=False)
hmm_regimes_merged_ticker = master_ticker[['time', 'ticker'] + state_cols].copy()
hmm_regimes_merged_ticker.to_csv(OUTPUT_DIR / 'hmm_regimes_merged_ticker.csv', index=False)
print("Đã lưu các file master của từng mã.")

split_dir = OUTPUT_DIR / 'splits_ticker'
split_dir.mkdir(parents=True, exist_ok=True)

df_sorted = master_ticker.sort_values('time').reset_index(drop=True)
train_end = '2019-12-31'
val_end = '2022-12-31'

df_train = df_sorted[df_sorted['time'] <= train_end].reset_index(drop=True)
df_val = df_sorted[(df_sorted['time'] > train_end) & (df_sorted['time'] <= val_end)].reset_index(drop=True)
df_test = df_sorted[df_sorted['time'] > val_end].reset_index(drop=True)

df_train.to_parquet(split_dir / 'train_set.parquet', index=False)
df_val.to_parquet(split_dir / 'val_set.parquet', index=False)
df_test.to_parquet(split_dir / 'test_set.parquet', index=False)
print("Đã phân chia các tập dữ liệu Splits cho từng mã thành công!")


Đã lưu các file master của từng mã.
Đã phân chia các tập dữ liệu Splits cho từng mã thành công!


## 9. Trực Quan Hóa Tương Tác Regimes Từng Mã (Interactive Ticker Regime Visualization)

In [16]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive_output
import pandas as pd

df_ticker = pd.read_parquet(OUTPUT_DIR / 'master_drl_ready_ticker.parquet')
df_ticker['time'] = pd.to_datetime(df_ticker['time'])

# Map unique labels to colors dynamically based on actual values in data (V2)
unique_labels = sorted(df_ticker['market_regime_label'].dropna().unique())
color_pool = ['#bbdefb', '#c8e6c9', '#fff9c4', '#ffcdd2', '#e0e0e0', '#d1c4e9', '#ffe0b2']
predefined = {
    'Bull': '#c8e6c9',          # Light Green
    'Bear': '#ffcdd2',          # Light Red
    'Sideways': '#e0e0e0',      # Light Grey
    'CalmBull': '#c8e6c9',      # Light Green
    'Euphoria': '#fff9c4',      # Light Yellow
    'Crisis': '#ffcdd2',        # Light Red
    'Tranquil': '#bbdefb',      # Light Grey
    'TranquilBull': '#bbdefb',      # Light Grey
    'Daily_Tier1': '#bbdefb',   # Light Blue
    'Daily_Tier2': '#c8e6c9',   # Light Green
    'Daily_Tier3': '#fff9c4',   # Light Yellow
    'Daily_Tier4': '#ffcdd2',   # Light Red
    'Daily_Tier5': '#e0e0e0',   # Light Grey
}
REGIME_COLORS = {}
for i, label in enumerate(unique_labels):
    if label in predefined:
        REGIME_COLORS[label] = predefined[label]
    else:
        REGIME_COLORS[label] = color_pool[i % len(color_pool)]

def plot_ticker_regimes(ticker, date_range):
    sub_df = df_ticker[
        (df_ticker['ticker'] == ticker) &
        (df_ticker['time'] >= date_range[0]) &
        (df_ticker['time'] <= date_range[1])
    ].copy().sort_values('time').reset_index(drop=True)
    
    if len(sub_df) == 0:
        print("Không có dữ liệu cho khoảng thời gian này.")
        return
        
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
    
    # Đồ thị giá đóng cửa
    ax1.plot(sub_df['time'], sub_df['close'], color='black', linewidth=1.5, label=f"{ticker} Close")
    ax1.set_title(f"Biểu Đồ Trạng Thái Ẩn HMM - Mã: {ticker}", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Giá Đóng Cửa (VND)", fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Tô màu nền theo trạng thái HMM
    regime_series = sub_df['market_regime_label']
    times = sub_df['time']
    
    for i in range(len(sub_df) - 1):
        reg = regime_series.iloc[i]
        color = REGIME_COLORS.get(reg, '#ffffff')
        ax1.axvspan(times.iloc[i], times.iloc[i+1], color=color, alpha=0.6)
        ax2.axvspan(times.iloc[i], times.iloc[i+1], color=color, alpha=0.6)
        
    # Đồ thị khối lượng giao dịch
    ax2.bar(sub_df['time'], sub_df['volume'], color='grey', alpha=0.6, label="Volume")
    ax2.set_ylabel("Khối Lượng", fontsize=12)
    ax2.set_xlabel("Thời Gian", fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    # Tạo legend tùy chỉnh cho regimes
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=color, edgecolor='none', label=label) 
        for label, color in REGIME_COLORS.items()
    ]
    ax1.legend(handles=legend_elements + [ax1.get_lines()[0]], loc='upper left')
    
    plt.tight_layout()
    plt.show()

# Thiết lập Widgets cho Jupyter
ticker_dropdown = widgets.Dropdown(
    options=sorted(df_ticker['ticker'].unique()),
    value='HPG',
    description='Mã Ticker:',
)

dates = sorted(df_ticker['time'].unique())
date_slider = widgets.SelectionRangeSlider(
    options=dates,
    index=(0, len(dates)-1),
    description='Khoảng Đo:',
    orientation='horizontal',
    layout={'width': '80%'}
)

ui = widgets.VBox([ticker_dropdown, date_slider])
out = interactive_output(plot_ticker_regimes, {'ticker': ticker_dropdown, 'date_range': date_slider})

display(ui, out)


Output()